In [1]:
%env GH_TOKEN YOUR_GITHUB_TOKEN

env: GH_TOKEN=YOUR_GITHUB_TOKEN


In [2]:
from semantic_release.cli.config import (
    GlobalCommandLineOptions,
    RawConfig,
    RuntimeContext,
)
from semantic_release.cli.util import load_raw_config_file, rprint
from git.repo.base import Repo
from pathlib import Path

DEFAULT_CONFIG_FILE = 'pyproject.toml'

repo = Repo(".", search_parent_directories=True)

config_path = Path(DEFAULT_CONFIG_FILE)
config_text = load_raw_config_file(config_path)

cli_options = GlobalCommandLineOptions(
        noop=False,
        verbosity=1,
        config_file=DEFAULT_CONFIG_FILE,
        strict=False,
    )

raw_config = RawConfig.model_validate(config_text)

runtime = RuntimeContext.from_raw_config(
    raw_config, repo=repo, global_cli_options=cli_options
)

In [3]:
repo

<git.repo.base.Repo '/Users/chiauhung/Documents/multi-semantic-release/.git'>

In [4]:
config_text

{'version_variables': ['pyproject.toml:tool.poetry.version'],
 'version_toml': ['pyproject.toml:tool.poetry.version'],
 'branch': 'main',
 'upload_to_pypi': False,
 'upload_to_vcs_release': True,
 'build_command': 'pip install poetry && poetry build'}

In [5]:
runtime

RuntimeContext(repo=<git.repo.base.Repo '/Users/chiauhung/Documents/multi-semantic-release/.git'>, commit_parser=<semantic_release.commit_parser.angular.AngularCommitParser object at 0x1087cdb10>, version_translator=VersionTranslator(tag_format=v{version}, prerelease_token=rc), major_on_zero=True, prerelease=False, assets=[], commit_author=<git.Actor "semantic-release <semantic-release>">, commit_message='{version}\n\nAutomatically generated by python-semantic-release', changelog_excluded_commit_patterns=(re.compile('(?P<version>.*)\n\nAutomatically generated by python-semantic-release'),), version_declarations=(<semantic_release.version.declaration.TomlVersionDeclaration object at 0x109792250>, <semantic_release.version.declaration.PatternVersionDeclaration object at 0x1079b88d0>), hvcs_client=<semantic_release.hvcs.github.Github object at 0x1079f1710>, changelog_file=PosixPath('/Users/chiauhung/Documents/multi-semantic-release/CHANGELOG.md'), ignore_token_for_push=False, template_env

---

In [6]:
from semantic_release.version import (
    Version,
    next_version,
    tags_and_versions,
)

repo = runtime.repo
parser = runtime.commit_parser
translator = runtime.version_translator
major_on_zero = runtime.major_on_zero
prerelease = False
assets = runtime.assets
commit_author = runtime.commit_author
commit_message = runtime.commit_message
major_on_zero = runtime.major_on_zero
build_command = runtime.build_command
opts = runtime.global_cli_options
changelog_excluded_commit_patterns = runtime.changelog_excluded_commit_patterns

new_version = next_version(
    repo=repo,
    translator=translator,
    commit_parser=parser,
    prerelease=prerelease,
    major_on_zero=major_on_zero,
)

new_version

Version(major=0, minor=1, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}')

In [9]:
from semantic_release.cli.commands.version import apply_version_to_source_files

files_with_new_version_written = apply_version_to_source_files(
        repo=repo,
        version_declarations=runtime.version_declarations,
        version=new_version,
        noop=opts.noop,
    )

all_paths_to_add = files_with_new_version_written + (assets or [])
print(all_paths_to_add)

repo.git.add(all_paths_to_add)

['pyproject.toml', 'pyproject.toml']


''

In [7]:
from semantic_release.changelog import ReleaseHistory, environment, recursive_render
from datetime import datetime

rh = ReleaseHistory.from_git_history(
        repo=repo,
        translator=translator,
        commit_parser=parser,
        exclude_commit_patterns=changelog_excluded_commit_patterns,
    )

commit_date = datetime.now()
try:
    rh = rh.release(
        new_version,
        tagger=commit_author,
        committer=commit_author,
        tagged_date=commit_date,
    )
except ValueError as ve:
    print(str(ve))

In [8]:
rh

<ReleaseHistory: 0 commits unreleased, 17 versions released>

In [33]:
from typing import ContextManager

def custom_git_environment() -> ContextManager[None]:
        """
        git.custom_environment is a context manager but
        is not reentrant, so once we have "used" it
        we need to throw it away and re-create it in
        order to use it again
        """
        return (
            nullcontext()
            if not commit_author
            else repo.git.custom_environment(
                GIT_AUTHOR_NAME=commit_author.name,
                GIT_AUTHOR_EMAIL=commit_author.email,
                GIT_COMMITTER_NAME=commit_author.name,
                GIT_COMMITTER_EMAIL=commit_author.email,
            )
        )

remote_url = runtime.hvcs_client.remote_url(
            use_token=not runtime.ignore_token_for_push
        )
active_branch = repo.active_branch.name

with custom_git_environment():
    # repo.git.commit(
    #     m=commit_message.format(version=new_version),
    #     date=int(commit_date.timestamp()),
    # )

    repo.git.tag("-a", new_version.as_tag(), m=new_version.as_tag())
    repo.git.push(remote_url, active_branch)
    # repo.git.push("--tags", remote_url, active_branch)

GitCommandError: Cmd('git') failed due to: exit code(1)
  cmdline: git push --tags https://*****@github.com/neurotichl/pyproject_test.git main
  stderr: 'To https://github.com/neurotichl/pyproject_test.git
 * [new tag]         v2.6.1 -> v2.6.1
 * [new tag]         v2.7.0 -> v2.7.0
 ! [rejected]        v2.3.2 -> v2.3.2 (already exists)
 ! [rejected]        v2.5.0 -> v2.5.0 (already exists)
error: failed to push some refs to 'https://github.com/neurotichl/pyproject_test.git'
hint: Updates were rejected because the tag already exists in the remote.'

In [35]:
from semantic_release.cli.common import (
    render_default_changelog_file,
    render_release_notes,
)

release = rh.released[new_version]
        # Use a new, non-configurable environment for release notes -
        # not user-configurable at the moment
release_note_environment = environment(template_dir=runtime.template_dir)
# changelog_context.bind_to_environment(release_note_environment)
release_notes = render_release_notes(
    template_environment=release_note_environment,
    version=new_version,
    release=release,
)
try:
    release_id = hvcs_client.create_or_update_release(
        tag=new_version.as_tag(),
        release_notes=release_notes,
        prerelease=new_version.is_prerelease,
    )
except Exception as e:
    log.exception(e)
    ctx.fail(str(e))

TemplateRuntimeError: No filter named 'commit_hash_url' found.

In [9]:
{v for _, v in tags_and_versions(repo.tags, translator)}

{Version(major=0, minor=0, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=0, minor=1, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=0, minor=2, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=0, minor=3, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=0, minor=4, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=1, minor=0, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=2, minor=0, patch=0, prerelease_token='rc', prerelease_revision=None, build_metadata='', tag_format='v{version}'),
 Version(major=2, minor=1, patch=0, prerelease_token='rc', prerelease_revision=None, build

In [7]:
all_git_tags_as_versions = tags_and_versions(repo.tags, translator)
all_full_release_tags_and_versions = [
    (t, v) for t, v in all_git_tags_as_versions if not v.is_prerelease
]
print(
    "Found %s full releases (excluding prereleases)",
    len(all_full_release_tags_and_versions),
)

DEFAULT_VERSION = "0.0.0"

# Default initial version
latest_full_release_tag, latest_full_release_version = next(
    iter(all_full_release_tags_and_versions),
    (None, translator.from_string(DEFAULT_VERSION)),
)

Found %s full releases (excluding prereleases) 17


In [8]:
print(latest_full_release_tag.name, repo.active_branch)

v2.7.0 main


In [11]:
repo.git.merge

<function git.cmd.Git.__getattr__.<locals>.<lambda>(*args, **kwargs)>

In [12]:
merge_bases = repo.merge_base(latest_full_release_tag.name, repo.active_branch)
merge_bases

[<git.Commit "a612dd20fd7879a1139ec33a508a6f09f5c5c250">]

In [13]:
repo.merge_base(repo.active_branch, repo.active_branch)

[<git.Commit "84a78a01facc53ee3a4aab5a99a7acc7e9f45dc7">]

In [17]:
from semantic_release.version.algorithm import _bfs_for_latest_version_in_history

if latest_full_release_tag is None:
    # Workaround - we can safely scan the extra commits on this
    # branch if it's never been released, but we have no other
    # guarantees that other branches exist
    print(
        "No full releases have been made yet, the default version to use is %s",
        latest_full_release_version,
    )
    merge_bases = repo.merge_base(repo.active_branch, repo.active_branch)
else:
    # Note the merge_base might be on our current branch, it's not
    # necessarily the merge base of the current branch with `main`
    print(
        "The last full release was %s, tagged as %r",
        latest_full_release_version,
        latest_full_release_tag,
    )
    merge_bases = repo.merge_base(latest_full_release_tag.name, repo.active_branch)
    
if len(merge_bases) > 1:
    raise NotImplementedError(
        "This branch has more than one merge-base with the "
        "latest version, which is not yet supported"
    )
    
merge_base = merge_bases[0]
if merge_base is None:
    str_tag_name = (
        "None" if latest_full_release_tag is None else latest_full_release_tag.name
    )
    raise ValueError(
        f"The merge_base found by merge_base({str_tag_name}, {repo.active_branch}) "
        "is None"
    )

latest_full_version_in_history = _bfs_for_latest_version_in_history(
    merge_base=merge_base,
    full_release_tags_and_versions=all_full_release_tags_and_versions,
)
print(
    "The last full version in this branch's history was %s",
    latest_full_version_in_history,
)

commits_since_last_full_release = (
    repo.iter_commits()
    if latest_full_version_in_history is None
    else repo.iter_commits(f"{latest_full_version_in_history.as_tag()}...")
)

# Step 4. Parse each commit since the last release and find any tags that have
# been added since then.
parsed_levels = set()
latest_version = latest_full_version_in_history or Version(
    0,
    0,
    0,
    prerelease_token=translator.prerelease_token,
    tag_format=translator.tag_format,
)

# N.B. these should be sorted so long as we iterate the commits in reverse order
for commit in commits_since_last_full_release:
    parse_result = commit_parser.parse(commit)
    if isinstance(parse_result, ParsedCommit):
        log.debug(
            "adding %s to the levels identified in commits_since_last_full_release",
            parse_result.bump,
        )
        parsed_levels.add(parse_result.bump)

The last full release was %s, tagged as %r 2.7.0 v2.7.0
The last full version in this branch's history was %s 2.7.0


NameError: name 'commit_parser' is not defined

In [18]:
commits_since_last_full_release = (
    repo.iter_commits()
    if latest_full_version_in_history is None
    else repo.iter_commits(f"{latest_full_version_in_history.as_tag()}...")
)

In [19]:
list(commits_since_last_full_release)

[<git.Commit "84a78a01facc53ee3a4aab5a99a7acc7e9f45dc7">,
 <git.Commit "29858e587e678cf0e8ea610d79fade26f4cbfb76">]